# 01 — Data exploration and sample selection

This notebook establishes which data the design can be built on. It inventories
the AIDev distribution, locates a human-authored comparison group, collects the
review data that group lacks, and determines which review measures the resulting
sample can support.

**Research objective.** Analyze agentic and human pull requests within the same
repositories, for the purpose of comparing agentic and human PRs, with respect
to the coverage and outcome of human review over the repository's agentic
history, from the viewpoint of open-source maintainers and practitioners
adopting coding agents, in the context of AIDev revision `68ed5f4b`, restricted
to repositories carrying at least ten PRs of each origin.

**RQ1.** Does the share of agent-authored PRs receiving human review change over
the course of a repository's agentic history?

**RQ2.** Is any such change specific to agent-authored PRs, or do human-authored
PRs in the same repositories follow the same trajectory?

**Positioning.** Yu et al. [ref] report a within-reviewer rise in approval rates
for agent-authored PRs alongside declining inline comment effort, and read this
as habituation. Their unit of analysis is the review: a PR that receives none
produces no observation. Withdrawal of oversight — reviewers ceasing to look
rather than looking less carefully — is therefore outside what their design can
measure, and the 76% of agentic PRs carrying no formal review (Section 3) are
absent from their sample by construction.

Two consequences shape this notebook. First, measuring review coverage requires
the PR as the unit, which in turn requires a repository-level rather than
reviewer-level design. Second, establishing that any drift is specific to
agentic PRs requires a paired comparison against human-authored PRs in the same
repositories; Yu et al. report this as infeasible at the reviewer level, where
only 26 of their reviewers carry sufficient volume of both origins in both
periods. Section 5 shows the pairing holds for 94% of repositories.

## 1. Dataset inventory

We first enumerate the tables actually published in the current AIDev
release, and confirm the availability of each table our design depends
on.

In [1]:
from huggingface_hub import list_repo_files
import pandas as pd

files = sorted(f for f in list_repo_files("hao-li/AIDev", repo_type="dataset")
               if f.endswith(".parquet"))

print("Available tables:")
for f in files:
    print(" ", f)


Available tables:
  all_pull_request.parquet
  all_repository.parquet
  all_user.parquet
  pr_comments.parquet
  pr_commit_details.parquet
  pr_commits.parquet
  pr_review_comments.parquet
  pr_reviews.parquet
  pull_request.parquet
  repository.parquet
  user.parquet


**Result.** The current release contains eleven tables, none of which
holds human-authored PRs: the `agent` column is populated for all 71,677
records in `pull_request`. Published work using AIDev reports a
`human_pull_request` table, which suggests the dataset has changed since
those studies. Section 2 examines its revision history.

## 2. Dataset versioning

The Hugging Face commit history shows a release labelled "Promote v4
(AIDev-2.7M, cutoff Nov 2025) to main" in August 2026. This is a major
revision: it roughly doubles the agentic subset, adds a sixth agent
(Google Jules), and removes several tables — among them
`human_pull_request`, `pr_task_type`, `pr_timeline`, and the issue
tables.

We therefore work from revision `68ed5f4b` (May 2026), the last state
preceding v4. This choice trades coverage for the availability of a
comparison group, and has the side benefit of matching the release on
which the studies we position against were conducted. It is recorded
here for reproducibility: a dataset that changes underneath an analysis
is itself a threat to validity, and results reported against AIDev are
not comparable across revisions unless the revision is stated.

In [2]:
from huggingface_hub import HfApi

api = HfApi()
commits = api.list_repo_commits("hao-li/AIDev", repo_type="dataset")

for c in commits[:30]:
    print(c.created_at.date(), c.commit_id[:8], c.title)

2026-08-31 c63c8a57 Update README.md
2026-08-21 a1a0e853 Promote v4 (AIDev-2.7M, cutoff Nov 2025) to main
2026-05-10 68ed5f4b Update README.md
2026-04-10 288c5aa7 Update README.md
2026-04-04 54aecb41 Add a paper using AIDEV dataset (#14)
2026-04-03 c5cff179 Update README.md
2026-03-18 f7870e7c Added `write_page_index=True`
2026-02-18 512e0701 Update README.md
2026-02-18 27cfa4e1 Add task categories and link paper (#12)
2026-02-04 ddfb6dd4 New paper contribution using AIDev (#11)
2026-01-26 8b6deb28 Update README.md
2026-01-18 24313fa8 Update README.md
2026-01-18 fc7fca59 Update README.md
2025-11-05 eee0408a Update README.md
2025-10-30 20cd3dfb Update README.md
2025-10-29 a2b41111 Update README.md
2025-10-29 6200a09f Create data_table.md
2025-10-28 1865ebc9 Update README.md
2025-10-28 4c65121f Update dataset
2025-10-16 7bb7ab5e Update README.md
2025-10-16 580def03 Update dataset
2025-10-14 aadd7db2 Update README.md
2025-08-25 07fdd9bb Update dataset
2025-08-23 94b62dbf Update README.md


In [3]:
for sha in ["68ed5f4b", "4c65121f", "580def03", "07fdd9bb"]:
    files = list_repo_files("hao-li/AIDev", repo_type="dataset", revision=sha)
    parquets = [f for f in files if f.endswith(".parquet")]
    has_human = any("human" in f for f in parquets)
    print(f"{sha}  human table: {has_human}  ({len(parquets)} fichiers)")
    if has_human:
        print("   ", parquets)

68ed5f4b  human table: True  (18 fichiers)
    ['all_pull_request.parquet', 'all_repository.parquet', 'all_user.parquet', 'human_pr_task_type.parquet', 'human_pull_request.parquet', 'issue.parquet', 'pr_comments.parquet', 'pr_commit_details.parquet', 'pr_commits.parquet', 'pr_review_comments.parquet', 'pr_review_comments_v2.parquet', 'pr_reviews.parquet', 'pr_task_type.parquet', 'pr_timeline.parquet', 'pull_request.parquet', 'related_issue.parquet', 'repository.parquet', 'user.parquet']
4c65121f  human table: True  (18 fichiers)
    ['all_pull_request.parquet', 'all_repository.parquet', 'all_user.parquet', 'human_pr_task_type.parquet', 'human_pull_request.parquet', 'issue.parquet', 'pr_comments.parquet', 'pr_commit_details.parquet', 'pr_commits.parquet', 'pr_review_comments.parquet', 'pr_review_comments_v2.parquet', 'pr_reviews.parquet', 'pr_task_type.parquet', 'pr_timeline.parquet', 'pull_request.parquet', 'related_issue.parquet', 'repository.parquet', 'user.parquet']
580def03  human 

In [4]:
REV = "68ed5f4b"
BASE = f"hf://datasets/hao-li/AIDev@{REV}"

agentic_df = pd.read_parquet(f"{BASE}/pull_request.parquet")
human_df   = pd.read_parquet(f"{BASE}/human_pull_request.parquet")
reviews_df = pd.read_parquet(f"{BASE}/pr_reviews.parquet")
comments_df = pd.read_parquet(f"{BASE}/pr_comments.parquet")

print("agentic:", agentic_df.shape)
print("human:  ", human_df.shape)

agentic: (33596, 14)
human:   (6618, 13)


## 3. Availability of the review measure

Our design requires review intensity on both agentic and human PRs.
`pr_reviews` and `pr_comments` are not labelled by authorship, so whether
they cover both populations has to be established by joining on PR
identifiers rather than assumed.

In [5]:
human_ids   = set(human_df.id)
agentic_ids = set(agentic_df.id)
review_ids  = set(reviews_df.pr_id)
comment_ids = set(comments_df.pr_id)

print("Human PRs with reviews: ",
      len(human_ids & review_ids), "/", len(human_ids))
print("Human PRs with comments:",
      len(human_ids & comment_ids), "/", len(human_ids))
print()
print("Agentic PRs with reviews: ",
      len(agentic_ids & review_ids), "/", len(agentic_ids))
print("Agentic PRs with comments:",
      len(agentic_ids & comment_ids), "/", len(agentic_ids))

Human PRs with reviews:  0 / 6618
Human PRs with comments: 0 / 6618

Agentic PRs with reviews:  8140 / 33596
Agentic PRs with comments: 12975 / 33596


**Result.** Review data covers agentic PRs only: 8,140 of 33,596 agentic
PRs carry at least one review and 12,975 carry at least one comment,
against zero of 6,618 human PRs. The `human_pull_request` table holds PR
metadata alone.

Two consequences follow. First, the measure central to our objective is
unavailable on the comparison group as distributed; we collect it via the
GitHub API (Section 4). Second, 76% of agentic PRs carry no formal review
— a substantive observation rather than missing data. Whether this is
treated as a zero or as an exclusion is an operationalisation choice with
material consequences, and one the study we position against does not
face: taking the review as its unit, PRs that receive none simply do not
enter its sample.

### 3.1 Locating inline review comments

`pr_review_comments` is not keyed on the pull request. Its
`pull_request_review_id` points at `pr_reviews.id`, so reaching agentic PRs
requires joining through the review table rather than directly — a detail the
published schema diagram does not make explicit.

In [6]:
prc = pd.read_parquet(f"{BASE}/pr_review_comments.parquet")
print(prc.columns.tolist())
print(len(set(agentic_df.id) & set(prc.pull_request_review_id)))

['id', 'pull_request_review_id', 'user', 'user_type', 'diff_hunk', 'path', 'position', 'original_position', 'commit_id', 'original_commit_id', 'body', 'pull_request_url', 'created_at', 'updated_at', 'in_reply_to_id']
0


In [7]:
# Via pr_reviews
rev_ids = set(reviews_df[reviews_df.pr_id.isin(agentic_df.id)].id)
print("via reviews:", prc.pull_request_review_id.isin(rev_ids).sum())

# Ou directement par l'URL
print(prc.pull_request_url.iloc[0])

via reviews: 19450
https://api.github.com/repos/micropython/micropython/pulls/17613


**Result.** 19,450 inline comments attach to agentic PRs once routed through
`pr_reviews`. The measure therefore exists on the agentic side; Section 4.4
collects its counterpart for human-authored PRs.

## 4. Collecting review data for human-authored PRs

The review measure is unavailable on the comparison group as distributed,
so we collect it ourselves. For each of the 6,618 human-authored PRs, we
retrieve formal reviews and issue comments through the GitHub REST API,
matching the fields already available for agentic PRs. Yu et al. report
an equivalent collection; ours is independent, and the two are not
expected to coincide exactly, since repositories continue to change
after the dataset snapshot.

Two calls per PR are required — `/pulls/{n}/reviews` and
`/issues/{n}/comments` — for roughly 13,200 requests against an
authenticated limit of 5,000 per hour. The collection therefore spans
several hours and is written incrementally, so that an interruption does
not discard completed work and the run can be resumed.

Requires `GITHUB_TOKEN` in `.env` (scope: `public_repo`).

### 4.1 Authentication

In [8]:
from dotenv import load_dotenv
import os, requests

load_dotenv("../.env", override=True)
TOKEN = os.environ["GITHUB_TOKEN"]

r = requests.get("https://api.github.com/rate_limit",
                 headers={"Authorization": f"Bearer {TOKEN}"})
print("Rate limit:", r.json()["rate"])

Rate limit: {'limit': 5000, 'used': 0, 'remaining': 5000, 'reset': 1789238306}


### 4.2 Comparison group structure

The human-authored table carries the same schema as the agentic one,
with `agent` set to `Human` throughout, so the two can be stacked
directly using that column as the grouping variable. Repository
references are given as API URLs, which serve as the base for collection.

The schemas match, but the populations are not symmetric: AIDev was
built outward from agent accounts, and the human table was assembled
separately. What the two tables share is PR metadata; everything beyond
that — reviews, comments, commit details — exists only on the agentic
side.

In [9]:
print("Columns:", human_df.columns.tolist())
print("\nOrigin label:")
print(human_df.agent.value_counts(dropna=False))
print("\nrepo_url format:", human_df.repo_url.iloc[0])
human_df.head(2)

Columns: ['id', 'number', 'title', 'user', 'user_id', 'state', 'created_at', 'closed_at', 'merged_at', 'repo_url', 'html_url', 'body', 'agent']

Origin label:
agent
Human    6618
Name: count, dtype: int64

repo_url format: https://api.github.com/repos/getsentry/sentry


,id,number,title,user,user_id,state,created_at,closed_at,merged_at,repo_url,html_url,body,agent
0,2336888723,85268,feat(aci): add automations index page,ameliahsu,55610339,closed,2025-02-14T19:04:59Z,2025-02-18T22:42:20Z,2025-02-18T22:42:19Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/85268,https://sentry-j41gpomr5.sentry.dev/automation...,Human
1,2447123365,89131,ref(insights): Make use of `<FeatureBadge>` fo...,ryan953,187460,closed,2025-04-08T23:29:50Z,2025-04-09T15:56:55Z,2025-04-09T15:56:54Z,https://api.github.com/repos/getsentry/sentry,https://github.com/getsentry/sentry/pull/89131,Using the premade component reduces an import ...,Human


In [10]:
import os, time, json, requests
from pathlib import Path

HEADERS = {"Authorization": f"Bearer {TOKEN}",
           "Accept": "application/vnd.github+json"}

OUT = Path("../data/human_pr_reviews.jsonl")
OUT.parent.mkdir(exist_ok=True)

done = set()
if OUT.exists():
    with OUT.open() as f:
        done = {json.loads(line)["pr_id"] for line in f}
print(f"{len(done)} PRs already collected")


def get(url):
    """GET with rate-limit handling."""
    while True:
        r = requests.get(url, headers=HEADERS)
        if r.status_code == 403 and "rate limit" in r.text.lower():
            reset = int(r.headers.get("X-RateLimit-Reset", time.time() + 60))
            wait = max(reset - time.time(), 0) + 5
            print(f"rate limited — sleeping {wait/60:.1f} min")
            time.sleep(wait)
            continue
        r.raise_for_status()
        return r.json()


todo = human_df[~human_df.id.isin(done)]

with OUT.open("a") as f:
    for i, row in enumerate(todo.itertuples(), 1):
        base = row.repo_url

        try:
            reviews  = get(f"{base}/pulls/{row.number}/reviews?per_page=100")
            comments = get(f"{base}/issues/{row.number}/comments?per_page=100")
        except requests.HTTPError as e:
            print(f"skip PR {row.id}: {e.response.status_code}")
            continue

        f.write(json.dumps({
            "pr_id": row.id,
            "repo": base.split("/repos/")[-1],
            "number": row.number,
            "n_reviews": len(reviews),
            "n_comments": len(comments),
            "reviewers": sorted({r["user"]["login"] for r in reviews if r.get("user")}),
            "review_states": [r["state"] for r in reviews],
            "commenters": sorted({c["user"]["login"] for c in comments if c.get("user")}),
        }) + "\n")
        f.flush()

        if i % 100 == 0:
            print(f"{i}/{len(todo)}")

print("done")

6424 PRs already collected
skip PR 2428115577: 404
skip PR 2513413649: 404
skip PR 2355300308: 404
skip PR 2486006659: 404
skip PR 2412901600: 404
skip PR 2465464455: 404
skip PR 2544110590: 404
skip PR 2427616889: 404
skip PR 2358334023: 404
skip PR 2591917397: 404
skip PR 2592855865: 404
skip PR 2425648879: 404
skip PR 2364787187: 404
skip PR 2483823196: 404
skip PR 2428059466: 404
skip PR 2448702145: 404
skip PR 2518007609: 404
skip PR 2456002199: 404
skip PR 2400483157: 404
skip PR 2483258537: 404
skip PR 2300139059: 404
skip PR 2519480153: 404
skip PR 2326990083: 404
skip PR 2319826569: 404
skip PR 2498655063: 404
skip PR 2484483431: 404
skip PR 2505788580: 404
skip PR 2476066957: 404
skip PR 2542668583: 404
skip PR 2435513480: 404
skip PR 2446305071: 404
skip PR 2499880325: 404
skip PR 2500329315: 404
skip PR 2477927624: 404
skip PR 2515383588: 404
skip PR 2441456618: 404
skip PR 2591512128: 404
skip PR 2517769935: 404
skip PR 2534810519: 404
skip PR 2529686187: 404
skip PR 24702

**Result.** Of the 6,618 human-authored PRs, 6,424 (97.1%) were
successfully retrieved. The remaining 194 returned HTTP 404, indicating
repositories that have since been deleted, renamed, or made private in
the eighteen months separating the dataset snapshot from our collection.

Attrition is not necessarily random: if losses concentrate in a small
number of repositories, those repositories drop out of the human group
while remaining in the agentic one, unbalancing the comparison. We
therefore check whether the failures are dispersed across repositories
or concentrated, and exclude affected repositories from both groups
where the latter holds.

In [11]:
collected = pd.read_json(OUT, lines=True)
missing = human_df[~human_df.id.isin(collected.pr_id)]

print(len(missing), "missing PRs")
print(missing.repo_url.value_counts().head(10))
print("number of missing repos", missing.repo_url.nunique())

194 missing PRs
repo_url
https://api.github.com/repos/antiwork/flexile        154
https://api.github.com/repos/RockChinQ/LangBot        19
https://api.github.com/repos/plexguide/Huntarr.io     11
https://api.github.com/repos/lunary-ai/lunary          6
https://api.github.com/repos/kiwicom/orbit             4
Name: count, dtype: int64
number of missing repos 5


In [12]:
missing_repos = set(missing.repo_url)
print(agentic_df[agentic_df.repo_url.isin(missing_repos)].shape)

(227, 14)


The 194 failures concentrate in 5 repositories, each having disappeared
in full rather than losing individual PRs. These repositories account for
227 agentic PRs. Retaining them would leave those PRs in the sample with
no human counterpart, biasing any within-repository comparison. We
therefore exclude the 5 repositories from both groups.

### 4.3 Second pass: inline review comments

The first collection retrieved issue comments — the general discussion
thread attached to a PR. The agentic side of AIDev, however, carries
inline review comments: those anchored to a specific line of the diff,
via `pr_review_comments`. These are distinct objects, and the reference
study we replicate measures the latter.

Comparing one against the other would be a construct error, so we run a
second pass over `/pulls/{n}/comments` for the human-authored PRs. The
same pass also records the commenter's account type, which AIDev
provides for agentic PRs (`user_type`) but which our first collection
omitted — without it, bot filtering could not be applied symmetrically
across the two groups.

Roughly 6,400 requests are required, one per retrieved PR.

In [13]:
OUT2 = Path("../data/human_pr_inline_comments.jsonl")

done2 = set()
if OUT2.exists():
    with OUT2.open() as f:
        done2 = {json.loads(line)["pr_id"] for line in f}
print(f"{len(done2)} PRs already collected")

# only PRs retrieved in the first pass
collected = pd.read_json(OUT, lines=True)
todo = human_df[human_df.id.isin(collected.pr_id) & ~human_df.id.isin(done2)]
print(f"{len(todo)} to go")

failed = []

with OUT2.open("a") as f:
    for i, row in enumerate(todo.itertuples(), 1):
        try:
            inline = get(f"{row.repo_url}/pulls/{row.number}/comments?per_page=100")
        except requests.HTTPError as e:
            failed.append({"pr_id": row.id, "status": e.response.status_code})
            continue

        f.write(json.dumps({
            "pr_id": row.id,
            "n_inline": len(inline),
            "n_words": sum(len(c["body"].split()) for c in inline if c.get("body")),
            "commenters": [
                {"login": c["user"]["login"], "type": c["user"]["type"]}
                for c in inline if c.get("user")
            ],
            "n_threads": len({c.get("in_reply_to_id") or c["id"] for c in inline}),
        }) + "\n")
        f.flush()

        if i % 200 == 0:
            print(f"{i}/{len(todo)}")

print(f"done — {len(failed)} failures")

6424 PRs already collected
0 to go
done — 0 failures


## 5. Sample feasibility

With both groups aligned on the same repositories, we can assess whether
the design is supportable. Our objective compares review intensity
between agentic and human PRs over the repository's agentic history.
Three conditions follow.

First, a repository must contain PRs of both origins — a repository with
agentic PRs alone contributes nothing to a comparison. Second, it must
contain enough of each for a temporal pattern to be estimable rather than
inferred from two or three points. Third, PRs of both origins must appear
in both halves of the period: a repository whose human PRs all predate
its agentic ones cannot support a paired within-repository test.

These conditions trade against one another: raising the per-repository
requirement sharpens the within-repository signal while shrinking the
number of repositories that qualify. Rather than fixing a threshold in
advance, we report the trade-off across candidate values and select from
it, revisiting the choice in a sensitivity analysis.

In [14]:
agentic_counts = agentic_df.groupby('repo_url').size()
human_counts   = human_df.groupby('repo_url').size()

both = pd.DataFrame({'agentic': agentic_counts, 'human': human_counts}).dropna()
print(f"Repositories with both origins: {len(both)}\n")

for k in [3, 5, 10, 20, 50]:
    sub = both[(both.agentic >= k) & (both.human >= k)]
    print(f"{k}+ of each: {len(sub):4d} repos, "
          f"{int(sub.agentic.sum()):6d} agentic, {int(sub.human.sum()):6d} human")

# retained sample, pinned at the primary threshold
THRESHOLD = 10
q = both[(both.agentic >= THRESHOLD) & (both.human >= THRESHOLD)]
kept = q.index
print(f"\nRetained at {THRESHOLD}+: {len(q)} repositories")

Repositories with both origins: 810

3+ of each:  376 repos,   8786 agentic,   6032 human
5+ of each:  262 repos,   8229 agentic,   5639 human
10+ of each:  130 repos,   6843 agentic,   4784 human
20+ of each:   75 repos,   5679 agentic,   4042 human
50+ of each:   25 repos,   3454 agentic,   2571 human

Retained at 10+: 130 repositories


**Result.** 810 repositories contain PRs of both origins. Requiring at least
ten of each retains 130 repositories, 6,843 agentic and 4,784 human PRs. Ten
observations per origin leave five on each side of the split, and 130 pairs
are ample for the signed-rank test that follows.

Tightening the requirement from three to ten reduces the repository count by
two thirds but discards only 22% of agentic PRs: the excluded repositories are
those contributing least to the comparison. We adopt ten as the primary
threshold and report twenty as a sensitivity check.

### 5.1 Pairability

The third condition is the one that constrains Yu et al.: applied to individual
reviewers, requiring volume of both origins in both periods leaves 26 of their
728 reviewers, and they report the paired agent-versus-human test as infeasible
on that basis. We check what the same requirement costs at the repository level.

The split follows their procedure: the midpoint is taken on the count of
observations rather than on elapsed time, so a repository whose activity is
concentrated in a few weeks is divided where its PRs divide, not where the
calendar does.

In [15]:
import numpy as np

stacked = pd.concat([
    agentic_df.assign(origin='agentic')[['repo_url', 'created_at', 'origin']],
    human_df.assign(origin='human')[['repo_url', 'created_at', 'origin']],
])
stacked['created_at'] = pd.to_datetime(stacked.created_at)
stacked = stacked.sort_values(['repo_url', 'created_at'])

# midpoint on observation count, per repository
rank = stacked.groupby('repo_url').cumcount()
size = stacked.groupby('repo_url').repo_url.transform('size')
stacked['half'] = np.where(rank < size // 2, 'early', 'late')

cover = stacked.groupby(['repo_url', 'origin', 'half']).size().unstack([1, 2])
pairable = cover.notna().all(axis=1)

for k in [5, 10, 20]:
    sub = both[(both.agentic >= k) & (both.human >= k)]
    n_ok = pairable.reindex(sub.index).fillna(False).sum()
    print(f"{k}+ of each: {len(sub):4d} qualify, {n_ok:4d} pairable "
          f"({n_ok / len(sub):.1%})")

5+ of each:  262 qualify,  218 pairable (83.2%)
10+ of each:  130 qualify,  123 pairable (94.6%)
20+ of each:   75 qualify,   73 pairable (97.3%)


**Result.** 122 of the 130 repositories (93.8%) carry PRs of both origins in
both halves of their history. The rate rises with the threshold (82.8% at five,
97.3% at twenty), as repositories with more PRs spread them more evenly over
the period.

This is where the repository level differs materially from the reviewer level.
A repository receives PRs of both origins throughout its history; an individual
reviewer need not. The pairing that Yu et al. report as infeasible therefore
holds here by construction, which is what makes a paired agent-versus-human
test available to this design and not to theirs.

### 5.2 Review volume

PR counts bound the unit of the test; they do not bound the measure. With 76%
of agentic PRs carrying no formal review, ten PRs may yield only two or three
reviews, and an approval rate computed on so few takes only a handful of
distinct values — the early/late difference would then be dominated by noise
rather than by any shift in behaviour.

Whether the approval-rate measure is available at this threshold is therefore a
separate question from whether the sample is large enough, and it decides which
operationalisation the analysis can carry. For reference, the reviewer at the
median of Yu et al.'s cohort compares 8 reviews against 9.

In [16]:
a_ids = set(agentic_df[agentic_df.repo_url.isin(kept)].id)
h_ids = set(human_df[human_df.repo_url.isin(kept)].id)

collected = pd.read_json(OUT, lines=True)
h = collected[collected.pr_id.isin(h_ids)]

print(f"{'':10s} {'PRs':>8s} {'reviews':>9s} {'comments':>9s}")
print(f"{'agentic':10s} {len(a_ids):8d} "
      f"{reviews_df.pr_id.isin(a_ids).sum():9d} "
      f"{comments_df.pr_id.isin(a_ids).sum():9d}")
print(f"{'human':10s} {len(h_ids):8d} "
      f"{int(h.n_reviews.sum()):9d} {int(h.n_comments.sum()):9d}")

# human reviews received by agentic PRs, per repository
a_rev = (reviews_df[reviews_df.user_type == "User"]
         .merge(agentic_df[['id', 'repo_url']], left_on='pr_id', right_on='id'))
per_repo = a_rev[a_rev.repo_url.isin(kept)].groupby('repo_url').size()

print("\nHuman reviews per repository (on agentic PRs)")
print(per_repo.describe().round(1))
for k in [10, 20, 40]:
    print(f"  {k}+ reviews: {(per_repo >= k).sum():3d} repos")

                PRs   reviews  comments
agentic        6843     12598     18328
human          4784      8839      8766

Human reviews per repository (on agentic PRs)
count    116.0
mean      68.6
std      111.0
min        1.0
25%        9.0
50%       24.5
75%       79.2
max      794.0
dtype: float64
  10+ reviews:  86 repos
  20+ reviews:  69 repos
  40+ reviews:  47 repos


**Result.** The retained sample carries 12,598 human reviews and 18,328
comments on its 6,843 agentic PRs, against 8,839 and 8,766 on 4,784 human
PRs. The agentic total is of the same order as the 11,429 reviews Yu et al.
observe, so the repository-level filtering does not cost the study its
review base.

Distribution is another matter. Human reviews on agentic PRs range from 1
to 794 per repository, with a median of 24.5 and a standard deviation
(111.0) exceeding the mean (68.6). At the first quartile, nine reviews
split into four and five per period — an approval rate computed on five
observations takes six distinct values, and its early/late difference
carries little information. Fourteen repositories drop out entirely,
having received no human review on any agentic PR.

Two measures therefore operate on two samples. Review coverage — the share
of agentic PRs receiving at least one human review — is computed over PRs
rather than reviews, is unaffected by this constraint, and runs on all 122
pairable repositories. Approval rate follows Yu et al.'s operationalisation
and is restricted to the 69 repositories carrying at least twenty human
reviews, leaving a minimum of ten per period. The reduction in pairs costs
statistical power, reported alongside the result.

The fourteen excluded repositories are not incidental to the question. A
repository where no agentic PR is ever reviewed is precisely the state that
a withdrawal of oversight would produce, and it is invisible to any measure
defined over reviews. That coverage retains these cases while approval rate
cannot is the clearest instance of what the choice of unit determines.

### 5.3 Reviewer turnover

Whether a repository-level drift reflects reviewers changing their behaviour
or reviewers being replaced is an empirical question, not one settled by the
fact that repositories are made of reviewers. Separating the two requires a
baseline: reviewers observed in both halves of a repository's history, whose
early behaviour can be held fixed

In [18]:
a_rev_s = a_rev.merge(agentic_df[['id', 'created_at']].rename(
    columns={'id': 'pr_id2'}), left_on='pr_id', right_on='pr_id2', how='left')
a_rev_s['submitted_at'] = pd.to_datetime(a_rev_s.submitted_at)
a_rev_s = a_rev_s[a_rev_s.repo_url.isin(kept)].sort_values(
    ['repo_url', 'submitted_at'])

# split each repository's review sequence at its midpoint
rk = a_rev_s.groupby('repo_url').cumcount()
sz = a_rev_s.groupby('repo_url').repo_url.transform('size')
a_rev_s['half'] = np.where(rk < sz // 2, 'early', 'late')

per_repo_rev = (a_rev_s.groupby(['repo_url', 'half'])
                .user.agg(set).unstack()).dropna()

turnover = pd.DataFrame({
    'n_early':   per_repo_rev.early.apply(len),
    'n_late':    per_repo_rev.late.apply(len),
    'n_overlap': [len(e & l) for e, l in
                  zip(per_repo_rev.early, per_repo_rev.late)],
})
turnover['retention'] = turnover.n_overlap / turnover.n_early

print(f"Repositories with reviewers in both halves: {len(turnover)}\n")
print(turnover.describe().round(2))
print(f"\nNo overlap at all:        {(turnover.n_overlap == 0).sum():3d}")
for m in [1, 3, 5]:
    print(f"  {m}+ overlapping reviewers: "
          f"{(turnover.n_overlap >= m).sum():3d} repos")

Repositories with reviewers in both halves: 109

       n_early  n_late  n_overlap  retention
count   109.00  109.00     109.00     109.00
mean      4.94    4.98       3.07       0.71
std       5.53    5.05       2.99       0.30
min       1.00    1.00       0.00       0.00
25%       2.00    2.00       1.00       0.50
50%       3.00    3.00       2.00       0.75
75%       6.00    6.00       4.00       1.00
max      39.00   27.00      15.00       1.00

No overlap at all:          6
  1+ overlapping reviewers: 103 repos
  3+ overlapping reviewers:  49 repos
  5+ overlapping reviewers:  22 repos


**Result.** 109 repositories have human reviewers in both halves of their
agentic history, with a median of 3 reviewers per period and 2 appearing in
both. Only 49 repositories carry three or more overlapping reviewers, and 22
carry five or more; median retention is 0.75.

This rules out one design we had considered. Separating behavioural change from
compositional change requires holding each reviewer's early behaviour fixed
while letting the reviewing population vary, and a baseline built on two
reviewers per repository cannot support that counterfactual. We therefore do not
attempt the decomposition, and note that a repository-level drift observed here
cannot be attributed to individual habituation rather than to turnover — a
limitation carried into the threats to validity rather than resolved.

Turnover is nonetheless moderate rather than wholesale: three quarters of early
reviewers are still reviewing in the late period at the median repository, so
the two populations being compared across periods are largely the same people.